# Model Architecture - ChatKasir
- Nama: Achmad Rif'an
- Bagian: AI-1 (Model Architect)

## 1. Imports Library

In [1]:
import os
import json
import pandas as pd
import numpy as np
import tensorflow as tf
from google.colab import drive
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, Embedding, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D, Bidirectional, LSTM
from tensorflow.keras.models import Model

# verifikasi versi
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"TensorFlow: {tf.__version__}")

# check GPU yang tersedia
print(f"GPU tersedia: {len(tf.config.list_physical_devices('GPU')) > 0}")

Pandas: 2.2.2
NumPy: 2.0.2
TensorFlow: 2.20.0
GPU tersedia: False


## 2. Data Loading

In [2]:
# Mount Google Drive
drive.mount('/content/drive')

# Definisikan Base Path
BASE_PATH = "/content/drive/MyDrive/ChatKasir"
DATA_PATH = f"{BASE_PATH}/assets/data"
TOKENIZER_PATH = f"{BASE_PATH}/assets/tokenizers"

# URL Dataset dari DS-1
url_food = "https://drive.google.com/uc?id=1xpoFjqAT9K0uwzSpVADm_EfKqG7dxVUI"
url_slang = "https://drive.google.com/uc?id=1G14C1qcqOp06Xs1HFiorE3Us_LLtaBs7"
url_sintetis = "https://drive.google.com/uc?id=17lFTivPH4BXEd6zoa-qjpUohNmh1ELdo"

# Fungsi membaca CSV dari Google Drive
# def load_gdrive_csv(url):
#    return pd.read_csv(url)

# Memuat ke dalam DataFrame
# df_food = load_gdrive_csv(url_food)
# df_slang = load_gdrive_csv(url_slang)
# df_sintetis = load_gdrive_csv(url_sintetis)
df_food = pd.read_csv(url_food)
df_slang = pd.read_csv(url_slang)
df_sintetis = pd.read_csv(url_sintetis)

print(f"Total data chat sintetis: {len(df_sintetis)} baris")
print(f"Total daftar menu: {len(df_food)} baris")

Mounted at /content/drive
Total data chat sintetis: 100500 baris
Total daftar menu: 18558 baris


In [3]:
# Tampilkan 5 baris pertama data food dan chat sintetis
display(df_food.head())
display(df_sintetis.head())

,name
0,abon
1,abon ayam
2,abon burger
3,abon cheese burger
4,abon goreng ayam


,input_text,product,quantity,price_satuan,pattern
0,kk mau pesen 10 daebak ken chicken wings [SEP]...,daebak ken chicken wings,10,3000,2
1,kk 3 nasi ayam betutu ya kak [SEP] noted kak n...,nasi ayam betutu,3,60000,2
2,bu mau pesen 6 indomie seafood [SEP] indomie s...,indomie seafood,6,37000,3
3,bg 9 chicken double dong [SEP] baik kak,chicken double,9,-1,1
4,bg bisa pesan 7 mie doer dong [SEP] oke mie do...,mie doer,7,42000,2


## 3. Tokenizer (WordPiece)

In [4]:
# Gabungkan teks chat sintetis dan nama makanan
semua_teks = df_sintetis['input_text'].astype(str).tolist() + df_food['name'].astype(str).tolist()

# Inisialisasi Tokenizer WordPiece dengan token [UNK] untuk kata tak dikenal
tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()  # Pisahkan kata berdasarkan spasi

# Atur Trainer: vocab 5000 untuk domain UMKM
# Special tokens:
# [PAD]: padding
# [UNK]: kata tak dikenal
# [SEP]: pemisah chat pembeli dan penjual
trainer = WordPieceTrainer(
    vocab_size=5000,
    special_tokens=["[PAD]", "[UNK]", "[SEP]"]
)

# Latih Tokenizer dengan teks gabungan
tokenizer.train_from_iterator(semua_teks, trainer)

# Simpan tokenizer ke file JSON
# os.makedirs("..\\assets\\tokenizers", exist_ok=True)
# tokenizer.save("..\\assets\\tokenizers\\tokenizer.json")
# print("Tokenizer disimpan di '..\\assets\\tokenizers\\tokenizer.json'")
os.makedirs(TOKENIZER_PATH, exist_ok=True)
tokenizer.save(f"{TOKENIZER_PATH}/tokenizer.json")
vocab_size = tokenizer.get_vocab_size()
print(f"Tokenizer disimpan. Vocab Size: {vocab_size}")

# Ambil ukuran kosakata akhir
vocab_size = tokenizer.get_vocab_size()
print(f"Ukuran Vocab: {vocab_size}")

# Tes kemampuan tokenizer menangani typo
tes_kalimat = "kk mau pesen 10 daebak ken chicken wings [SEP] baik kak daebak ken chicken wings rp3rb satuan totalnya rp30rb"
hasil_tes = tokenizer.encode(tes_kalimat)

print(f"\n===HASIL TES TOKENIZER===")
print(f"Kalimat asli: {tes_kalimat}")
print(f"Dipecah menjadi tokens: {hasil_tes.tokens}")
print(f"Diubah ke ID angka: {hasil_tes.ids}")


Tokenizer disimpan. Vocab Size: 5000
Ukuran Vocab: 5000

===HASIL TES TOKENIZER===
Kalimat asli: kk mau pesen 10 daebak ken chicken wings [SEP] baik kak daebak ken chicken wings rp3rb satuan totalnya rp30rb
Dipecah menjadi tokens: ['kk', 'mau', 'pesen', '10', 'daebak', 'ken', 'chicken', 'wings', '[SEP]', 'baik', 'kak', 'daebak', 'ken', 'chicken', 'wings', 'rp3rb', 'satuan', 'totalnya', 'rp30rb']
Diubah ke ID angka: [308, 125, 175, 172, 1379, 1318, 144, 358, 2, 164, 84, 1379, 1318, 144, 358, 1690, 177, 105, 1309]


## 4. Menghitung Max Length

In [5]:
# Hitung panjang token dari setiap baris di dataset chat sintetis
panjang_semua_teks = [len(tokenizer.encode(teks).ids) for teks in df_sintetis['input_text'].astype(str)]

# Cari yang paling panjang
max_length = max(panjang_semua_teks)

print(f"Panjang kalimat maksimal di dataset (max_length): {max_length}")

# Membulatkan max_length ke 64
optimal_max_length = 64 if max_length < 64 else max_length
print(f"Max Length optimal yang akan digunakan: {optimal_max_length}")

Panjang kalimat maksimal di dataset (max_length): 31
Max Length optimal yang akan digunakan: 64


## 5. Arsitektur Transformer

In [6]:
# Arsitektur "Dual-Brain" (Transformer + BiLSTM)
class TransformerEncoder(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super(TransformerEncoder, self).__init__(**kwargs)
        self.supports_masking = True
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([Dense(ff_dim, activation="relu"), Dense(embed_dim)])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training=False, mask=None):
        padding_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.int32) if mask is not None else None
        attn_output = self.att(inputs, inputs, attention_mask=padding_mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Bangun arsitektur multi-task dengan Transformer
def model_transformer(vocab_size, max_length, num_product_tags):
    # Parameter dasar Transformer
    embed_dim = 128  # Dimensi embedding (representasi kata)
    num_heads = 4   # Jumlah head attention
    ff_dim = 256    # Dimensi feed forward

    # Input Layer berupa ID token
    inputs = Input(shape=(max_length,), name="input_ids")

    # Shared Layer
    x = Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=True)(inputs)  # Ubah ID jadi vektor
    x = TransformerEncoder(embed_dim, num_heads, ff_dim)(x)  # Masukkan ke Transformer Encoder
    x = TransformerEncoder(embed_dim, num_heads, ff_dim)(x)

    x_pooled = GlobalAveragePooling1D()(x)  # Pooling: meringkas kalimat jadi 1 vektor

    # Cabang 1: Produk (Bi-LSTM Optimized)
    branch_product = Bidirectional(LSTM(64, return_sequences=True))(x)
    branch_product = Dense(64, activation='relu')(branch_product)
    output_product = Dense(num_product_tags, activation='softmax', name="product_tags")(branch_product)

    # Cabang 2: Quantity (regresi, prediksi angka kuantitas)
    branch_quantity = Dense(32, activation='relu')(x_pooled)
    output_quantity = Dense(1, activation='relu', name="quantity")(branch_quantity)

    # Cabang 3: Price (regresi, prediksi harga satuan)
    branch_price = Dense(32, activation='relu')(x_pooled)
    output_price = Dense(1, activation='relu', name="price")(branch_price)

    return Model(inputs=inputs, outputs=[output_product, output_quantity, output_price])

# Merancang model dengan 3 tag produk
# 'O' (Bukan produk), 'B-PROD' (Awal produk), 'I-PROD' (Lanjutan produk)
NUM_PRODUCT_TAGS = 3
model = model_transformer(
    vocab_size=vocab_size,
    max_length=optimal_max_length,
    num_product_tags=NUM_PRODUCT_TAGS
    )

# Ringkasan arsitektur model
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_ids           │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 64, 128)   │    640,000 │ input_ids[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 64)        │          0 │ input_ids[0][0]   │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encoder │ (None, 64, 128)   │    330,240 │ embedding[0][0],  │
│ (TransformerEncode… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, 64, 128)   │    330,240 │ transformer_enco… │
│ (TransformerEncode… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 64, 128)   │     98,816 │ transformer_enco… │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ transformer_enco… │
│ (GlobalAveragePool… │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64, 64)    │      8,256 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 32)        │      4,128 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 32)        │      4,128 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product_tags        │ (None, 64, 3)     │        195 │ dense_4[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ quantity (Dense)    │ (None, 1)         │         33 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ price (Dense)       │ (None, 1)         │         33 │ dense_6[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,416,069 (5.40 MB)

 Trainable params: 1,416,069 (5.40 MB)

 Non-trainable params: 0 (0.00 B)

## 6. Dataset Preparation

In [7]:
# Kamus sederhana untuk memetakan Tag BIO ke angka
# 0 = Bukan produk (O)
# 1 = Awal kata produk (B-PROD)
# 2 = Lanjutan kata produk (I-PROD)
TAG2ID = {'O': 0, 'B-PROD': 1, 'I-PROD': 2}

def siapkan_data_latih(df, tokenizer, max_len):
    X_input_ids = []
    Y_product_tags = []
    Y_quantity = []
    Y_price = []

    for index, row in df.iterrows():
        # Ambil data dari baris dataset
        teks_chat = str(row['input_text'])
        nama_produk = str(row['product'])

        # Tangani jika ada nilai kosong, berikan default
        qty = float(row['quantity']) if pd.notnull(row['quantity']) else 0.0
        # Jika harga kosong, biarkan -1 agar nanti diabaikan oleh MaskedPriceLoss kita
        price = float(row['price_satuan']) if pd.notnull(row['price_satuan']) else -1.0

        # Ubah teks menjadi angka ID menggunakan Tokenizer
        encoded_teks = tokenizer.encode(teks_chat).ids
        encoded_produk = tokenizer.encode(nama_produk).ids

        # Siapkan label 'O' (0) sebanyak jumlah token teks
        tags = [TAG2ID['O']] * len(encoded_teks)

        # Cari posisi kata produk di dalam kalimat untuk memberikan tag BIO
        panjang_produk = len(encoded_produk)
        for i in range(len(encoded_teks) - panjang_produk + 1):
            # Jika potongan token teks sama dengan token produk
            if encoded_teks[i:i+panjang_produk] == encoded_produk:
                tags[i] = TAG2ID['B-PROD'] # Kata pertama
                for j in range(1, panjang_produk):
                    tags[i+j] = TAG2ID['I-PROD'] # Kata selanjutnya
                break # Berhenti mencari jika sudah ketemu

        # Padding
        if len(encoded_teks) > max_len:
            # Potong jika kepanjangan
            encoded_teks = encoded_teks[:max_len]
            tags = tags[:max_len]
        else:
            # Tambah [PAD] jika kependekan
            selisih = max_len - len(encoded_teks)
            id_pad = tokenizer.token_to_id("[PAD]")

            encoded_teks = encoded_teks + [id_pad] * selisih
            tags = tags + [TAG2ID['O']] * selisih # Padding tag tetap dihitung 'O'

        # Simpan ke daftar utama
        X_input_ids.append(encoded_teks)
        Y_product_tags.append(tags)
        Y_quantity.append(qty)
        Y_price.append(price)

    # Ubah menjadi array numpy agar bisa dibaca TensorFlow
    return np.array(X_input_ids), np.array(Y_product_tags), np.array(Y_quantity), np.array(Y_price)

# Panggil fungsinya menggunakan dataset kita
X, Y_prod, Y_qty, Y_price = siapkan_data_latih(df_sintetis, tokenizer, optimal_max_length)

print("Data sudah siap untuk dilatih")
print(f"Bentuk Input (X): {X.shape}")
print(f"Bentuk Target Produk (Y_prod): {Y_prod.shape}")

Data sudah siap untuk dilatih
Bentuk Input (X): (100500, 64)
Bentuk Target Produk (Y_prod): (100500, 64)


## 7. Split Dataset (Training, Validation, Testing)

In [8]:
# Pisahkan data Training dulu (80%), sisa 20% simpan di variabel sementara (temp)
X_train, X_temp, Y_prod_train, Y_prod_temp, Y_qty_train, Y_qty_temp, Y_price_train, Y_price_temp = train_test_split(
    X, Y_prod, Y_qty, Y_price, test_size=0.20, random_state=42
)

# Bagi sisa 20% data sama rata (50-50) untuk Validation dan Testing
X_val, X_test, Y_prod_val, Y_prod_test, Y_qty_val, Y_qty_test, Y_price_val, Y_price_test = train_test_split(
    X_temp, Y_prod_temp, Y_qty_temp, Y_price_temp, test_size=0.50, random_state=42
)

print("Hasil Pembagian Data:")
print(f"1. Training (80%): {len(X_train)} baris")
print(f"2. Validation (10%): {len(X_val)} baris")
print(f"3. Testing (10%): {len(X_test)} baris")

# Simpan semua array ke dalam satu file kompresi numpy (.npz)
# os.makedirs("..\\assets\\data", exist_ok=True)
# lokasi_simpan = "..\\assets\\data\\dataset_chatkasir.npz"

os.makedirs(DATA_PATH, exist_ok=True)
lokasi_simpan = f"{DATA_PATH}/dataset_chatkasir.npz"

np.savez(lokasi_simpan,
         X_train=X_train, Y_prod_train=Y_prod_train, Y_qty_train=Y_qty_train, Y_price_train=Y_price_train,
         X_val=X_val, Y_prod_val=Y_prod_val, Y_qty_val=Y_qty_val, Y_price_val=Y_price_val,
         X_test=X_test, Y_prod_test=Y_prod_test, Y_qty_test=Y_qty_test, Y_price_test=Y_price_test)

print(f"\nData berhasil disimpan di: {lokasi_simpan}")

Hasil Pembagian Data:
1. Training (80%): 80400 baris
2. Validation (10%): 10050 baris
3. Testing (10%): 10050 baris

Data berhasil disimpan di: /content/drive/MyDrive/ChatKasir/assets/data/dataset_chatkasir.npz


## 8. Menyimpan Konfigurasi

In [9]:
config_model = {
    "vocab_size": vocab_size,
    "max_length": optimal_max_length,
    "num_product_tags": NUM_PRODUCT_TAGS,
    "embed_dim": 128,
    "num_heads": 4,
    "ff_dim": 256
}

# lokasi_config = "..\\assets\\data\\model_config.json"
lokasi_config = f"{DATA_PATH}/model_config.json"
with open(lokasi_config, "w") as f:
    json.dump(config_model, f, indent=4)

print(f"Konfigurasi arsitektur berhasil disimpan di: {lokasi_config}")

Konfigurasi arsitektur berhasil disimpan di: /content/drive/MyDrive/ChatKasir/assets/data/model_config.json
